# Final BART Meeting Summarizer


In [1]:

import os
import re
import glob
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "facebook/bart-large-cnn"
MODEL_FILE = "bart_model.pt"

TRANSCRIPT_DIR = os.path.join("outputs", "assemblyai", "transcripts")
SUMMARY_DIR = os.path.join("outputs", "BART", "assemblyai")

os.makedirs(SUMMARY_DIR, exist_ok=True)

MAX_KEY_POINTS = 5
MAX_DECISIONS = 8
MAX_ACTIONS = 8
MAX_DATES = 8

print("Transcript folder:", os.path.abspath(TRANSCRIPT_DIR))
print("Summary folder:", os.path.abspath(SUMMARY_DIR))
print("Model:", os.path.abspath(MODEL_FILE))


Transcript folder: /Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai/transcripts
Summary folder: /Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/BART/assemblyai
Model: /Users/manish/Documents/KU/SEM 5/DL/Mini Project/bart_model.pt


In [2]:

if not os.path.isfile(MODEL_FILE):
    raise FileNotFoundError(f"Model file not found: {os.path.abspath(MODEL_FILE)}")

if not os.path.isdir(TRANSCRIPT_DIR):
    raise FileNotFoundError(f"Transcript folder not found: {os.path.abspath(TRANSCRIPT_DIR)}")

transcript_files = sorted(glob.glob(os.path.join(TRANSCRIPT_DIR, "*.txt")))

if not transcript_files:
    raise FileNotFoundError("No transcript TXT files were found.")

print("Found", len(transcript_files), "transcript(s).")


Found 25 transcript(s).


In [3]:

device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu"))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

state_dict = torch.load(MODEL_FILE, map_location=device)
model.load_state_dict(state_dict)
model.to(device)
model.eval()

print("Fine-tuned BART loaded successfully.")
print("Device:", device)


[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Fine-tuned BART loaded successfully.
Device: cpu


In [4]:

def clean(text):
    return re.sub(r"\s+", " ", str(text)).strip()

def normalize(text):
    return re.sub(r"[^a-z0-9 ]", " ", clean(text).lower()).strip()

def similarity(a, b):
    aa = set(normalize(a).split())
    bb = set(normalize(b).split())
    if not aa or not bb:
        return 0.0
    return len(aa & bb) / len(aa | bb)

def dedupe(items, threshold=0.68):
    result = []
    for item in items:
        item = clean(item)
        if not item:
            continue
        if any(similarity(item, old) >= threshold for old in result):
            continue
        result.append(item)
    return result

def parse_transcript(text):
    records = []
    for raw_line in text.splitlines():
        line = re.sub(r"^\s*\[[^\]]+\]\s*", "", raw_line)
        line = clean(line)
        if not line:
            continue

        match = re.match(r"^([^:]{1,70}):\s*(.+)$", line)
        if match:
            speaker = clean(match.group(1))
            body = clean(match.group(2))
        else:
            speaker = ""
            body = line

        if len(body.split()) >= 4:
            records.append({"speaker": speaker, "body": body})

    return records


In [5]:
DECISION_HEADER_RE = re.compile(
    r"\b(?:the|our)\s+(?:(?:final|main|budget)\s+)?decisions?\s+(?:are|is|were|was)\b|"
    r"\bthe\s+(?:final\s+)?decision\s+(?:is|was|will be)\b", re.I
)

DECISION_DIRECT_RE = re.compile(
    r"^(?:we(?:[’']ll|\s+will)\s+(?:use|adopt|keep|go with|proceed with|retain|include|exclude|schedule|target|set|select)\b|"
    r"let[’']?s\s+(?:use|adopt|keep|go with|proceed with|retain|include|exclude|schedule|target|set|select|configure|standardize)\b|"
    r"we\s+(?:have\s+)?(?:decided|agreed|approved|confirmed|finalized)\b)", re.I
)

REQUIREMENT_RE = re.compile(
    r"\b(?:should|needs?\s+to|must|has\s+to|have\s+to|need\s+to|is\s+required\s+to|are\s+required\s+to|required\s+to)\b", re.I
)
CAPABILITY_RE = re.compile(r"^(?:I|we)\s+(?:can|could)\b", re.I)

DECISION_SPLIT_RE = re.compile(
    r",\s*(?:and\s+)?(?=(?:run|keep|retain|include|exclude|hold|use|adopt|review|add|remove|rotate|test|compare|assign|alert|conduct|target|show|reduce|submit|monitor|apply|schedule|select|set|proceed|go with|configure|standardize|require|limit|start|refresh|perform)\b)|"
    r"\s+and\s+(?=(?:run|keep|retain|include|exclude|hold|use|adopt|review|add|remove|rotate|test|compare|assign|alert|conduct|target|show|reduce|submit|monitor|apply|schedule|select|set|proceed|go with|configure|standardize|require|limit|start|refresh|perform)\b)", re.I
)

def is_question(text): return bool(QUESTION_RE.search(clean(text)))
def is_commitment(text): return bool(COMMITMENT_RE.match(clean(text)))
def is_request(text): return bool(REQUEST_RE.match(clean(text)))
def is_decision(text): return bool(DECISION_HEADER_RE.search(clean(text)) or DECISION_DIRECT_RE.match(clean(text)))


In [6]:
QUESTION_RE = re.compile(r"\?$|^(?:can|could|would|should|shall|do|does|did|is|are|will|what|when|where|why|how|may|might)\b", re.I)
COMMITMENT_RE = re.compile(r"^(?:I\s+(?:will|shall|am going to|plan to|intend to)\b|I['’]ll\b)", re.I)
REQUEST_RE = re.compile(r"^(?:please|can you|could you|would you)\b", re.I)
CAPABILITY_RE = re.compile(r"^(?:I|we)\s+(?:can|could|may|might)\b", re.I)

def clean_decision(text):
    text = clean(text)
    header = DECISION_HEADER_RE.search(text)
    if header:
        text = text[header.end():]
        text = re.sub(r"^\s*:??\s*(?:to\s+)?", "", text, flags=re.I)
    text = re.sub(r"^we\s+(?:have\s+)?(?:decided|agreed|approved|confirmed|finalized)\s+(?:that\s+)?", "", text, flags=re.I)
    text = re.sub(r"^(?:we(?:[’']ll|\s+will)|let[’']?s)\s+", "", text, flags=re.I)
    return clean(text).rstrip("?.! ")

def split_decision_text(text):
    text = clean_decision(text)
    parts = DECISION_SPLIT_RE.split(text)
    return [clean(part).strip(" ,") for part in parts if len(clean(part).split()) >= 3]

def extract_decisions(records):
    decisions = []
    for record in records:
        body = clean(record["body"])
        if not body or is_question(body) or is_request(body) or not is_decision(body):
            continue
        decisions.extend(split_decision_text(body))
    cleaned = []
    for item in decisions:
        item = clean(item)
        if not item:
            continue
        duplicate = False
        for old in cleaned:
            a, b = normalize(item), normalize(old)
            shorter_overlap = len(set(a.split()) & set(b.split())) / max(1, min(len(set(a.split())), len(set(b.split()))))
            same_frequency = (
                ("metadata" in a and "metadata" in b and "four" in a and "four" in b and "hour" in a and "hour" in b)
            )
            if a in b or b in a or similarity(item, old) >= 0.62 or shorter_overlap >= 0.70 or same_frequency:
                duplicate = True
                break
        if not duplicate:
            cleaned.append(item)
    return cleaned[:MAX_DECISIONS]

def is_non_keypoint_sentence(text):
    text = clean(text)
    lower = text.lower()

    if is_question(text) or is_request(text) or is_commitment(text):
        return True
    if CAPABILITY_RE.match(text):
        return True

    planning = (
        "we should", "we need", "we have to", "we must",
        "let's", "lets", "could we", "can we",
        "i think we should", "i think we need"
    )
    if lower.startswith(planning):
        return True

    conversational = (
        "i agree", "that makes sense", "sounds good",
        "okay", "ok", "right", "yes", "no",
        "thanks", "thank you"
    )
    if lower.startswith(conversational):
        return True

    return False


In [7]:

def strip_commitment(body):
    body = clean(body)
    body = re.sub(
        r"^I\s+(?:will|shall|am going to|plan to|intend to)\s+",
        "",
        body,
        flags=re.I
    )
    body = re.sub(r"^I['’]ll\s+", "", body, flags=re.I)
    return clean(body).rstrip("?.! ")

def extract_actions(records):
    actions = []
    seen = set()

    for record in records:
        body = clean(record["body"])

        if not body or is_question(body) or is_request(body):
            continue

        if not is_commitment(body):
            continue

        task = strip_commitment(body)

        if len(task.split()) < 3:
            continue

        speaker = clean(record["speaker"]) or "Team"
        key = normalize(f"{speaker} {task}")

        if key in seen:
            continue

        seen.add(key)
        actions.append(f"{speaker} — {task}")

    return dedupe(actions, threshold=0.84)[:MAX_ACTIONS]


In [8]:
DATE_RE = re.compile(
    r"\b(?:next|this|coming)\s+(?:monday|tuesday|wednesday|thursday|friday|saturday|sunday)"
    r"(?:\s+at\s+\d{1,2}(?::\d{2})?\s*(?:am|pm))?"
    r"(?:\s+(?:morning|afternoon|evening))?\b|"
    r"\b(?:monday|tuesday|wednesday|thursday|friday|saturday|sunday)"
    r"(?:\s+at\s+\d{1,2}(?::\d{2})?\s*(?:am|pm)|\s+(?:morning|afternoon|evening))\b|"
    r"\b(?:today|tomorrow|yesterday)(?:\s+(?:morning|afternoon|evening))?\b|"
    r"\b(?:this|next|coming)\s+(?:week|month)\b|"
    r"\b\d{1,2}[/-]\d{1,2}(?:[/-]\d{2,4})?\b|"
    r"\b(?:jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may|jun(?:e)?|jul(?:y)?|aug(?:ust)?|sep(?:tember)?|oct(?:ober)?|nov(?:ember)?|dec(?:ember)?)\s+\d{1,2}(?:,\s*\d{4})?\b",
    re.I
)

DATE_TRIGGER_RE = re.compile(
    r"\b(?:by|before|until|deadline|due|scheduled\s+for|schedule(?:d)?|target(?:ed)?|"
    r"review|meeting|release|test|demo|submission|delivery|launch|rehearsal|go-live|"
    r"workshop|pilot|training|session|appointment|hold|conduct|submit|complete|finish|"
    r"prepare|update|configure|verify|deploy|implement|run|send|circulate|"
    r"i\s+(?:will|shall|am going to|plan to|intend to)|i['’]ll)\b", re.I
)

def extract_date_context(body, match):
    # Work with the local clause containing the date instead of the whole sentence.
    left = body.rfind(",", 0, match.start())
    right = body.find(",", match.end())

    if left < 0:
        left = 0
    else:
        left += 1
    if right < 0:
        right = len(body)

    clause = clean(body[left:right].strip(" ,:-"))

    # If the clause is too short, also inspect the immediately preceding clause.
    if len(clause.split()) < 2 and left > 0:
        prev_left = body.rfind(",", 0, left - 1)
        prev_left = 0 if prev_left < 0 else prev_left + 1
        clause = clean(body[prev_left:right].strip(" ,:-"))

    clause = re.sub(
        r"^(?:yes|okay|ok|right|so|and|then|the following)\s*,?\s*",
        "", clause, flags=re.I
    )

    # Remove speech/commitment prefixes.
    clause = re.sub(
        r"^(?:i\s+(?:will|shall|am going to|plan to|intend to)|i['’]ll|let[’']?s)\s+",
        "", clause, flags=re.I
    ).strip()

    # Remove the date itself; what remains should be the scheduled activity.
    clause = DATE_RE.sub("", clause, count=1).strip(" ,:-")

    clause = re.sub(
        r"^(?:at|on|by|before|until|for)\s+", "", clause, flags=re.I
    ).strip(" ,:-")

    clause = re.sub(r"\b(?:is|are|was|were|will be)\s*$", "", clause, flags=re.I).strip(" ,:-")
    clause = re.sub(r"\s+", " ", clause)

    if not clause or len(clause.split()) < 2:
        return None

    if len(clause.split()) > 10:
        clause = " ".join(clause.split()[-10:])

    return clause

def extract_dates(records):
    items = []

    for record in records:
        body = clean(record["body"])
        if not body or not DATE_TRIGGER_RE.search(body):
            continue

        if re.match(
            r"^(?:good morning|good afternoon|good evening|today everyone|"
            r"today we are|today we're|the agenda|the discussion)\b",
            body, re.I
        ):
            continue

        speaker = clean(record["speaker"]) or "Team"

        for match in DATE_RE.finditer(body):
            phrase = clean(match.group(0))
            context = extract_date_context(body, match)
            if not context:
                continue

            # Skip incomplete or purely conversational contexts.
            if re.search(r"^(?:the following|scheduled item|current target|current release target)$", context, re.I):
                continue

            # Require a concrete task/schedule word in the local context.
            if not re.search(
                r"\b(?:review|meeting|deadline|launch|release|test|demo|rehearsal|training|workshop|pilot|"
                r"submit|prepare|finish|complete|update|configure|verify|deploy|implement|run|schedule|hold|"
                r"conduct|send|circulate|target|training|checklist|build|dataset|report|assessment)\b",
                context, re.I
            ):
                continue

            items.append(f"{phrase} — {speaker}: {context}")

    # Prefer one clean entry per date phrase when several transcript lines repeat the same schedule.
    items = dedupe(items, threshold=0.88)
    return items[:MAX_DATES]


In [9]:
def is_factual_keypoint(body):
    text = clean(body)
    lower = text.lower()

    if len(text.split()) < 6:
        return False

    if is_non_keypoint_sentence(text):
        return False

    if re.search(r"^(?:i|we)\s+(?:can|could|will|shall|am going to|plan to|intend to)\b", text, re.I):
        return False

    if is_decision(text):
        return False

    if REQUIREMENT_RE.search(text):
        return False

    # Capability or commitment language anywhere near the beginning is not a factual key point.
    if re.match(r"^(?:i|we)\s+(?:can|could|will|shall|am going to|plan to|intend to|should|need to|must)\b", text, re.I):
        return False

    decision_like = re.compile(
        r"^(?:to summarize,\s*)?(?:we\s+(?:are|will|shall)\s+(?:using|adopting|keeping|going with)|"
        r"the\s+(?:current|final)\s+(?:approach|choice|plan)\s+is)\b", re.I
    )
    if decision_like.search(text):
        return False

    return True

def keypoint_score(body, frequency):
    words = normalize(body).split()
    informative = sum(
        1 for word in words
        if len(word) >= 4 and word not in {
            "this", "that", "with", "from", "have", "will",
            "were", "been", "there", "their", "about"
        }
    )
    numbers = sum(1 for word in words if any(ch.isdigit() for ch in word))
    return (
        min(len(words), 30) * 0.8
        + informative * 1.2
        + numbers * 2.0
        + min(frequency.get(normalize(body), 1), 3)
    )

def select_keypoint_sources(records, decisions):
    frequency = {}
    for record in records:
        body = clean(record["body"])
        key = normalize(body)
        if key:
            frequency[key] = frequency.get(key, 0) + 1

    candidates = []
    for record in records:
        body = clean(record["body"])
        if not is_factual_keypoint(body):
            continue
        if any(similarity(body, decision) >= 0.45 for decision in decisions):
            continue
        candidates.append(body)

    candidates = dedupe(candidates, threshold=0.68)
    candidates.sort(key=lambda text: keypoint_score(text, frequency), reverse=True)

    selected = []
    for candidate in candidates:
        if any(similarity(candidate, old) >= 0.55 for old in selected):
            continue
        selected.append(candidate)
        if len(selected) >= MAX_KEY_POINTS:
            break

    return selected

def normalize_keypoint_output(text):
    text = clean(text).strip("•-* ")
    if not text:
        return text

    numeric_start = re.match(r"^(\d+(?:[.,]\d+)*)(\s+)(.+)$", text)
    if numeric_start:
        number, spacing, rest = numeric_start.groups()
        rest = rest[:1].lower() + rest[1:] if rest else rest
        return number + spacing + rest

    for i, ch in enumerate(text):
        if ch.isalpha():
            return text[:i] + ch.upper() + text[i + 1:]

    return text

def flan_keypoint(statement):
    statement = normalize_keypoint_output(statement)

    if len(statement.split()) <= 18:
        return statement

    # BART-large-CNN is already a summarization model: feed the statement
    # directly instead of an instruction prompt (base BART echoes instructions).
    inputs = tokenizer(statement, return_tensors="pt", truncation=True, max_length=256).to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=45,
            num_beams=4,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    result = clean(tokenizer.decode(output[0], skip_special_tokens=True)).strip("•-* ")

    if not result:
        return statement

    if is_non_keypoint_sentence(result) or is_decision(result):
        return statement

    if similarity(result, statement) < 0.20:
        return statement

    source_words = set(normalize(statement).split())
    generated_words = set(normalize(result).split())
    important_numbers = {w for w in source_words if any(ch.isdigit() for ch in w)}

    if important_numbers and not important_numbers.issubset(generated_words):
        return statement

    return normalize_keypoint_output(result.rstrip("?.! ") + ".")

def extract_key_points(records, decisions):
    sources = select_keypoint_sources(records, decisions)
    result = [flan_keypoint(source) for source in sources]
    return [item for item in result if not is_non_keypoint_sentence(item) and not is_decision(item)]


In [10]:

def build_minutes(transcript, source_name):
    records = parse_transcript(transcript)

    decisions = extract_decisions(records)
    actions = extract_actions(records)
    dates = extract_dates(records)
    key_points = extract_key_points(records, decisions)

    key_points = [
        normalize_keypoint_output(item)
        for item in dedupe(key_points, threshold=0.78)
    ]

    lines = [
        "MEETING MINUTES",
        "=" * 60,
        f"Source: {source_name}",
        "",
        "KEY POINTS",
        "-" * 20
    ]

    lines.extend(f"• {item}" for item in key_points)
    if not key_points:
        lines.append("• No clear factual key points identified.")

    lines += ["", "DECISIONS", "-" * 20]
    lines.extend(f"• {item}" for item in decisions)
    if not decisions:
        lines.append("• No explicit decisions identified.")

    lines += ["", "ACTION ITEMS", "-" * 20]
    lines.extend(f"• {item}" for item in actions)
    if not actions:
        lines.append("• No explicit action commitments identified.")

    lines += ["", "IMPORTANT DATES / DEADLINES", "-" * 30]
    lines.extend(f"• {item}" for item in dates)
    if not dates:
        lines.append("• No explicit dates or deadlines identified.")

    return "\n".join(lines) + "\n"


In [11]:

generated_files = []

for transcript_path in transcript_files:
    source_name = os.path.basename(transcript_path)
    base_name = os.path.splitext(source_name)[0]
    output_path = os.path.join(SUMMARY_DIR, base_name + "_summary.txt")

    with open(transcript_path, "r", encoding="utf-8", errors="ignore") as f:
        transcript = f.read()

    minutes = build_minutes(transcript, source_name)

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(minutes)

    generated_files.append(output_path)
    print("OVERWRITTEN:", os.path.basename(output_path))

print()
print("Generation complete.")
print("Summary files written:", len(generated_files))


[transformers] Both `max_new_tokens` (=45) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/Users/manish/Documents/KU/.venv3.13/lib/python3.13/site-packages/transformers/generation/utils.py:1700: UserWarning: Unfeasible length constraints: `min_length` (56) is larger than the maximum possible length (46). Generation will stop at the defined maximum length. You should decrease the minimum length and/or increase the maximum length.
  warnings.warn(
[transformers] Both `max_new_tokens` (=45) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=45) and `max_length`(=142) seem to have been set. `max_new_tokens` will t

OVERWRITTEN: EN2002a_summary.txt


[transformers] Both `max_new_tokens` (=45) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=45) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=45) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=45) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

OVERWRITTEN: ES2004a_summary.txt


[transformers] Both `max_new_tokens` (=45) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=45) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=45) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=45) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

...

OVERWRITTEN: meeting_conversation_7_summary.txt


[transformers] Both `max_new_tokens` (=45) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=45) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=45) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=45) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

OVERWRITTEN: meeting_conversation_8_summary.txt


[transformers] Both `max_new_tokens` (=45) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=45) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


OVERWRITTEN: meeting_conversation_9_summary.txt

Generation complete.
Summary files written: 25


In [12]:

for output_path in generated_files:
    with open(output_path, "r", encoding="utf-8") as f:
        content = f.read()

    print("\n" + "=" * 75)
    print(os.path.basename(output_path))
    print("=" * 75)
    print(content)



EN2002a_summary.txt
MEETING MINUTES
Source: EN2002a.txt

KEY POINTS
--------------------
• If you double-click on a particular topic or something, then it goes to sort of like a default. So it'll be like all the cases of that topic, either whatever we decide the default to be.
• I don't know what happens when you double-click, actually. We could, for example, have a little menu that pops up. I think, so yeah, when you left-click it, you can.
• Yeah, so I don't know if you've looked at the search query language, but it's like a $a text Yeah, yeah, because that's the guy who declares a variable. Yeah, and then.
• "I'm not aiming for anything. I'm just, you know, I go into the lab and I say, right, am I going to change today? And it kind of just, it just develops,".
• Yeah, do you want to, you know, just get together and think about the different types of search we could possibly do, and then we could discuss it next meeting? Which options we really want to include.

DECISIONS
----------

Run all cells to generate the meeting summaries.
